# SER - Experiment 06: overnight regularisation sweep

**Diagnosis.** The base model reaches 98.9% training accuracy against 61.8%
validation accuracy, with validation loss bottoming out at epoch 4. It is not
under-powered, it is under-constrained:

| | |
|---|---|
| `Flatten(9,472) -> Dense(512)` | **4,850,176 params = 66% of the model** |
| `DROPOUT_CONV / DENSE` | 0.20 / 0.30 - light for a 37-point gap |
| L2 weight decay | **none anywhere** |
| Augmentation | 1.34x (8,756 -> 11,700) |

Every lever is untouched. This notebook pulls them.

## Rules

1. **The test set is never touched.** Every candidate is scored on the
   **validation** split. The winner gets exactly one test evaluation later,
   in a separate run.
2. **This does not alter the reproduction.** `base` remains 57.95% in the
   report; anything found here is a separate, clearly-labelled improvement.
3. Novelties are **off** for all candidates, so the effect measured is
   regularisation alone.

**Attach:** the four corpora + `ser-feature-cache`.
**Accelerator: GPU.** Roughly 2-2.5 hours; runs headless.

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import numpy as np
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Set Settings -> Accelerator -> GPU before running."

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
DATA_ROOT = "/kaggle/working/ser/data"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    src = find_canonical(target)
    assert src is not None, f"MISSING INPUT for {name}: no '{target}' found"
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
import config
from data_loader import build_metadata, split_metadata
from augmentation import plan_augmentation
from features import build_feature_matrix, df_to_items
from utils import StreamScalers, set_seed

config.CACHE_DIR = "/kaggle/working/features_cache"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

# Re-use the published cache where the fingerprints match (1.34x augmentation
# and the val split); the 3x variant is extracted fresh below.
staged = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".npz"):
            shutil.copy(os.path.join(root, f), config.CACHE_DIR)
            staged += 1
print(f"staged {staged} cache files")

meta = build_metadata(strict=True)
assert len(meta) == 12162
train_df, val_df, test_df = split_metadata(meta)
set_seed(config.RANDOM_SEED)

# --- 1.34x augmentation (the base-paper budget, already cached) ------------
items_1x = plan_augmentation(train_df, emotion_aware=False)
train_1x = build_feature_matrix(items_1x, desc="train_uniform")

# --- 3x augmentation (fresh extraction, ~20 min) ---------------------------
ORIG_TARGET = config.TARGET_TRAIN_SIZE
config.TARGET_TRAIN_SIZE = 3 * len(train_df)
items_3x = plan_augmentation(train_df, emotion_aware=False)
config.TARGET_TRAIN_SIZE = ORIG_TARGET
train_3x = build_feature_matrix(items_3x, desc="train_uniform3x")

val_feats = build_feature_matrix(df_to_items(val_df), desc="val")

print()
print(f"train 1.34x : {train_1x['mfcc'].shape[0]:,} rows")
print(f"train 3x    : {train_3x['mfcc'].shape[0]:,} rows")
print(f"validation  : {val_feats['mfcc'].shape[0]:,} rows")
print("TEST SET NOT LOADED - it stays sealed for this sweep.")

In [ ]:
def prepare(train_feats):
    """Fit per-stream scalers on this training set and transform train+val."""
    sc = StreamScalers().fit(train_feats)
    x_tr = sc.transform(train_feats)
    x_va = sc.transform(val_feats)
    y_tr = tf.keras.utils.to_categorical(train_feats["y"], config.NUM_CLASSES)
    y_va = tf.keras.utils.to_categorical(val_feats["y"], config.NUM_CLASSES)
    return sc, x_tr, y_tr, x_va, y_va


PREPARED = {"1x": prepare(train_1x), "3x": prepare(train_3x)}
print("scalers fitted for both augmentation budgets")

In [ ]:
# Each candidate changes ONE more thing than the previous. Novelties stay off
# throughout, so what is measured is regularisation alone.
CANDIDATES = [
    dict(tag="base",              data="1x", lr=1e-3, patience=10, kw={}),
    dict(tag="gap",               data="1x", lr=1e-3, patience=10,
         kw=dict(head="gap")),
    dict(tag="reg",               data="1x", lr=1e-3, patience=10,
         kw=dict(dropout_conv=0.35, dropout_dense=0.55, l2=1e-4)),
    dict(tag="gap_reg",           data="1x", lr=1e-3, patience=10,
         kw=dict(head="gap", dropout_conv=0.35, dropout_dense=0.55, l2=1e-4)),
    dict(tag="gap_reg_aug3",      data="3x", lr=1e-3, patience=12,
         kw=dict(head="gap", dropout_conv=0.35, dropout_dense=0.55, l2=1e-4)),
    dict(tag="gap_reg_aug3_lr",   data="3x", lr=5e-4, patience=15,
         kw=dict(head="gap", dropout_conv=0.35, dropout_dense=0.55, l2=1e-4)),
]

for c in CANDIDATES:
    print(f"  {c['tag']:20s} data={c['data']:3s} lr={c['lr']:.0e} "
          f"{c['kw']}")

In [ ]:
from model import build_model

RESULTS_PATH = "/kaggle/working/sweep_results.json"
results = []
if os.path.exists(RESULTS_PATH):
    results = json.load(open(RESULTS_PATH))
done = {r["tag"] for r in results}

started = time.time()

for c in CANDIDATES:
    if c["tag"] in done:
        print(f"[skip] {c['tag']} already done")
        continue

    sc, x_tr, y_tr, x_va, y_va = PREPARED[c["data"]]
    tf.keras.backend.clear_session()
    set_seed(config.RANDOM_SEED)

    model, _ = build_model(use_afw=False, use_mstc=False, **c["kw"])
    model.compile(optimizer=tf.keras.optimizers.Adam(c["lr"]),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    print(f"\n{'=' * 72}")
    print(f"[run] {c['tag']}   {model.count_params():,} params   "
          f"train={x_tr[0].shape[0]:,}   ({(time.time()-started)/60:.0f} min "
          f"elapsed)")
    print(f"{'=' * 72}")

    hist = model.fit(
        x_tr, y_tr, validation_data=(x_va, y_va),
        epochs=config.EPOCHS, batch_size=config.BATCH_SIZE, verbose=2,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=c["patience"],
                restore_best_weights=True, verbose=1),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=config.REDUCE_LR_FACTOR,
                patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR,
                verbose=0),
        ])

    h = hist.history
    best_i = int(np.argmin(h["val_loss"]))
    rec = {
        "tag": c["tag"],
        "params": int(model.count_params()),
        "val_accuracy": float(h["val_accuracy"][best_i]),
        "val_loss": float(h["val_loss"][best_i]),
        "train_accuracy": float(h["accuracy"][best_i]),
        "gap": float(h["accuracy"][best_i] - h["val_accuracy"][best_i]),
        "best_epoch": best_i + 1,
        "epochs_run": len(h["val_loss"]),
        "data": c["data"], "lr": c["lr"], "kw": {k: str(v) for k, v
                                                 in c["kw"].items()},
    }
    results.append(rec)
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

    model.save(os.path.join(config.RUNS_DIR, f"sweep_{c['tag']}.keras"))
    print(f"\n  -> val_acc {rec['val_accuracy']*100:.2f}%   "
          f"train_acc {rec['train_accuracy']*100:.2f}%   "
          f"gap {rec['gap']*100:.1f} pts   best epoch {rec['best_epoch']}")

print(f"\nsweep complete in {(time.time()-started)/60:.0f} min")

In [ ]:
BASE_VAL = None
rows = sorted(results, key=lambda r: -r["val_accuracy"])
for r in results:
    if r["tag"] == "base":
        BASE_VAL = r["val_accuracy"]

print("=" * 84)
print("  REGULARISATION SWEEP - VALIDATION ONLY (test set untouched)")
print("=" * 84)
print(f"  {'candidate':22s} {'val acc':>8s} {'train':>8s} {'gap':>7s} "
      f"{'params':>11s} {'epoch':>6s}  delta")
print("  " + "-" * 80)
for r in rows:
    d = "" if BASE_VAL is None else f"{(r['val_accuracy']-BASE_VAL)*100:+6.2f}"
    print(f"  {r['tag']:22s} {r['val_accuracy']*100:7.2f}% "
          f"{r['train_accuracy']*100:7.2f}% {r['gap']*100:6.1f} "
          f"{r['params']:11,} {r['best_epoch']:6d}  {d}")
print("=" * 84)

if BASE_VAL is not None and rows:
    best = rows[0]
    print()
    print(f"  WINNER: {best['tag']}  "
          f"{best['val_accuracy']*100:.2f}% validation "
          f"({(best['val_accuracy']-BASE_VAL)*100:+.2f} points vs base)")
    print(f"  Overfitting gap: {best['gap']*100:.1f} pts "
          f"(base was {[r for r in results if r['tag']=='base'][0]['gap']*100:.1f})")
    print()
    print("  NEXT: evaluate this configuration on the test set ONCE.")
    print("  Do not pick a winner by test accuracy - that is what keeps the")
    print("  number honest.")

In [ ]:
for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)
print("output:", sorted(os.listdir("/kaggle/working")))